# Python for Economics and Finance
## Scientific Library
### 4 Pandas: 3 Data Processing Example
### Shutao Cao
#### 2026-01-30

In [14]:
import numpy as np
import pandas as pd
pd.__version__
pd.set_option("mode.copy_on_write", True)

Below is an example of doing the following:
- Downloading Table 14100287 from Statistics Canada website.
- Cleaning data.
- Pivoting data.
- Saving data to local disk.

## Download data from websites
Many organizations that produce data sets provide web [application programming interface (API)](https://en.wikipedia.org/wiki/API). Web API allows us to write computer programs in Python or another language to download data files to local computer.

A partial list of organization provide web API:
- [Statistics Canada](https://www.statcan.gc.ca/en/start)
- [FRED](https://fred.stlouisfed.org/docs/api/fred/)
- World Bank
- International Monetary Fund
- U.S. [Bureau of Labor Statistics](https://www.bls.gov/bls/api_features.htm)
- Yahoo! Finance
- [World Trade Organization (WTO)](https://apiportal.wto.org/)

As an example, we download the quarterly data of labor market condition from Statistics Canada website, specifically, Table 14100287. The data file is in csv format and compressed, linked here: https://www150.statcan.gc.ca/n1/tbl/csv/14100287-eng.zip.

We use the Python libraries `requests`, `zipfile`, and `io`

In [ ]:
import requests as rq
import pandas as pd
import numpy as np
import zipfile
# 14100287: Labour force characteristics by age group, monthly, seasonally adjusted and unadjusted.
cansimid = '14100287'
url_to_data = 'https://www150.statcan.gc.ca/n1/tbl/csv/'+cansimid+'-eng.zip'
r = rq.get( url_to_data )
# r is now a Response object.

In [16]:
#r.headers
r.headers['Content-Type']

'application/zip'

Requests.get() returns binary data when the remote url is a zipped file (as we had in the above). We can do any of the following:
1. Save the zipped file directly to local disk.
2. Unzip the file and save unzipped file to local disk.
3. Unzip the file and process the unzipped file using Pandas.

In [17]:
# 1. Save requested content to local disk
with open(cansimid+'.zip', 'wb') as zf:
        zf.write(r.content)

To unzip the returned content r.content, we need to convert the Response object to a BytesIO object. A BytesIO object is a binary stream using a bytes buffer stored in memory (e.g., the r.content we obtained).

In [18]:
import io
buffer_data = io.BytesIO( r.content )
buffer_data

In [19]:
# Declare the buffer to be a zipped file
zippedData = zipfile.ZipFile( buffer_data )
print(zippedData.namelist())
csvFilename = zippedData.namelist()[ 0 ]
print(csvFilename)

['14100287.csv', '14100287_MetaData.csv']
14100287.csv


In [20]:
# 2. we can now extract (unzip) the file and save it to local disk
# Note, this extract the file in memory, not the zipped file saved in local disk
zippedData.extract(csvFilename, path=None, pwd=None)

'/home/xcst/teaching/2026wi_3210/notebooksIpynb/14100287.csv'

In [21]:
# How large is the csv file in local disk?
import os
print(os.path.getsize(csvFilename))

1164470697


In [22]:
# 3. Or, we can open zipped file to DataFrame, default csv encoding is utf-8
# Again, here we open the zipped file (which is a csv file) in memory, not from local disk
try:
    cansimTab = pd.read_csv(zippedData.open( csvFilename ), low_memory=False)
except UnicodeDecodeError:
    cansimTab = pd.read_csv(zippedData.open( csvFilename ), low_memory=False, encoding='ISO-8859-1',quoting=0)

cansimTab.columns

Index(['REF_DATE', 'GEO', 'DGUID', 'Labour force characteristics', 'Gender',
       'Age group', 'Statistics', 'Data type', 'UOM', 'UOM_ID',
       'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS',
       'SYMBOL', 'TERMINATED', 'DECIMALS'],
      dtype='object')

In [23]:
!ls *.zip

14100287.zip		  econ3210Assignment3.zip   LFS202401.zip
econ3210_assignment1.zip  econ3210_assignment4.zip  lfsQtr.csv.zip
econ3210Assignment2.zip   econ3210TakeHomeExam.zip


In [26]:
# 3. Alternatively, if opening the zipped file from local disk
zippedData = zipfile.ZipFile("14100287.zip",'r')
print(zippedData.namelist())
csvFilename = zippedData.namelist()[ 0 ]
print(csvFilename)
try:
    cansimTab = pd.read_csv(zippedData.open( csvFilename ), low_memory=False)
except UnicodeDecodeError:
    cansimTab = pd.read_csv(zippedData.open( csvFilename ), low_memory=False, encoding='ISO-8859-1',quoting=0)

cansimTab.columns

['14100287.csv', '14100287_MetaData.csv']
14100287.csv


Index(['REF_DATE', 'GEO', 'DGUID', 'Labour force characteristics', 'Gender',
       'Age group', 'Statistics', 'Data type', 'UOM', 'UOM_ID',
       'SCALAR_FACTOR', 'SCALAR_ID', 'VECTOR', 'COORDINATE', 'VALUE', 'STATUS',
       'SYMBOL', 'TERMINATED', 'DECIMALS'],
      dtype='object')

In [28]:
cansimTab.loc[:,['REF_DATE','GEO','Gender', 'Age group','Labour force characteristics','VALUE', 'VECTOR']]


,REF_DATE,GEO,Gender,Age group,Labour force characteristics,VALUE,VECTOR
0,1976-01,Canada,Total - Gender,15 years and over,Population,16852.4,v2062809
1,1976-01,Canada,Total - Gender,15 years and over,Population,16852.4,v2064888
2,1976-01,Canada,Total - Gender,15 to 64 years,Population,15015.9,v21580997
3,1976-01,Canada,Total - Gender,15 to 64 years,Population,15015.9,v21580998
4,1976-01,Canada,Total - Gender,15 to 24 years,Population,4509.9,v2062836
...,...,...,...,...,...,...,...
5394595,2025-12,British Columbia,Women+,55 years and over,Employment rate,30.1,v2064887
5394596,2025-12,British Columbia,Women+,55 years and over,Employment rate,30.1,v2066966
5394597,2025-12,British Columbia,Women+,55 years and over,Employment rate,0.9,v101889677
5394598,2025-12,British Columbia,Women+,55 years and over,Employment rate,0.6,v101889678


## Data Processing: case study
We use the downloaded data file on Canadian labor market condition as an example.

The data contains employment, unemployment, and labor force participation, both levels and rates. In Table 141000287, the raw data is a long form. Each variable is identified by VECTOR.

The long form is a vertical stack of all values of all variables, into one column VALUE.

Data processing can depend on individual users, I would like to make it a table-like form:
- Each row is the data of all variables in a month.
- Each column is the data of one variable in all months.

This case study demonstrates some useful Pandas methods.

### Column headers

In [29]:
# change column names
cansimTab.rename(columns={'Labour force characteristics':'laborvar','Age group':'ageGroup'}, inplace=True)
# use lower case for all column names
cansimTab.columns = [c.lower() for c in cansimTab.columns]
# remove blank spaces before and after a column name, in case it may have
cansimTab.columns = cansimTab.columns.str.strip()
# remove space, everywhere in a column name (better the strip())
cansimTab.columns = cansimTab.columns.str.replace(' ', '')
cansimTab.columns

Index(['ref_date', 'geo', 'dguid', 'laborvar', 'gender', 'agegroup',
       'statistics', 'datatype', 'uom', 'uom_id', 'scalar_factor', 'scalar_id',
       'vector', 'coordinate', 'value', 'status', 'symbol', 'terminated',
       'decimals'],
      dtype='object')

### Data selection
Table 14100287 contains seasonally unadjusted data, which are less of interest in economics (though could be useful in machine learning). We drop them.

The table also contains data of labor market condition in provinces, which we will also drop.

The table also contains data of labor market condition for various age groups, we want to keep data for age group "15 years and older".

In [30]:
cansimTab.datatype.unique()

array(['Seasonally adjusted', 'Unadjusted', 'Trend-cycle'], dtype=object)

In [31]:
cansimTab.geo.unique()

array(['Canada', 'Newfoundland and Labrador', 'Prince Edward Island',
       'Nova Scotia', 'New Brunswick', 'Quebec', 'Ontario', 'Manitoba',
       'Saskatchewan', 'Alberta', 'British Columbia'], dtype=object)

In [32]:
cansimTab.agegroup.unique()

array(['15 years and over', '15 to 64 years', '15 to 24 years',
       '15 to 19 years', '20 to 24 years', '25 years and over',
       '25 to 54 years', '55 years and over', '55 to 64 years'],
      dtype=object)

In [33]:
cansimTab.statistics.unique()

array(['Estimate', 'Standard error of estimate',
       'Standard error of month-to-month change',
       'Standard error of year-over-year change'], dtype=object)

In [34]:
# selecting data: keep only GEO=Canada; And Seasonally adjusted data
cansimTab2 = cansimTab[cansimTab.geo.isin(['Canada']) & cansimTab.datatype.isin(['Seasonally adjusted']) & cansimTab.statistics.isin(['Estimate']) & cansimTab.agegroup.isin(['15 years and over'])]

### Create mnemonics
Vector names, for example v2062809, are not informative, we would want to use mnemonics. An example, can be unempRate for the unemployment rate.

Two ways to create mnemonics:
1. Use documentation for vector names, which works but tedious, as we have to do this one by one.
2. Use information in the columns.

I choose the second method. Create mnemonic for each variable by sex. Ultimately, I create a new column that contains the mnemonics.

In [35]:
cansimTab2.laborvar.unique()

array(['Population', 'Labour force', 'Employment', 'Full-time employment',
       'Part-time employment', 'Unemployment', 'Unemployment rate',
       'Participation rate', 'Employment rate'], dtype=object)

In [38]:
# create variable mnemonics
# First, create a new column sex2 that will be part of mnemonics
import collections
import functools
sexName = ['Both gender', 'Males', 'Females']
sexKey = ['BothSex','Male','Female']
sexDict = collections.OrderedDict(zip(sexName,sexKey))
print(sexDict)
# use replace method to replace values
cansimTab2['gender2'] = cansimTab2['gender'].replace(sexDict)
cansimTab2[['gender','gender2']].head(100)

OrderedDict({'Both gender': 'BothSex', 'Males': 'Male', 'Females': 'Female'})


,gender,gender2
0,Total - Gender,Total - Gender
18,Men+,Men+
36,Women+,Women+
54,Total - Gender,Total - Gender
99,Men+,Men+
...,...,...
27446,Women+,Women+
27474,Total - Gender,Total - Gender
27519,Men+,Men+
27564,Women+,Women+


In [39]:
# second, create another new column laborvar2, holding part of mnemonics
laborName = list(cansimTab2.laborvar.unique())
"""
[Population', 'Labour force', 'Employment', 'Full-time employment','Part-time employment ',
'Unemployment', 'Unemployment rate', 'Participation rate', 'Employment rate']
"""
laborKey =['population','labForce', 'empl', 'emplFull','emplPart','unempl', 'unemplRate','participRate','emplRate']
laborDict = collections.OrderedDict(zip(laborName,laborKey))
print(laborDict)
# use map method to replace values
cansimTab2['laborvar2'] = cansimTab2['laborvar'].map(laborDict)
cansimTab2[['laborvar','laborvar2']].head(100)

OrderedDict({'Population': 'population', 'Labour force': 'labForce', 'Employment': 'empl', 'Full-time employment': 'emplFull', 'Part-time employment': 'emplPart', 'Unemployment': 'unempl', 'Unemployment rate': 'unemplRate', 'Participation rate': 'participRate', 'Employment rate': 'emplRate'})


,laborvar,laborvar2
0,Population,population
18,Population,population
36,Population,population
54,Labour force,labForce
99,Labour force,labForce
...,...,...
27446,Part-time employment,emplPart
27474,Unemployment,unempl
27519,Unemployment,unempl
27564,Unemployment,unempl


In [41]:
# now create a new column that holds the mnemonics
cansimTab2.loc[:,'laborvar3'] = cansimTab2.loc[:,'laborvar2']+ cansimTab2.loc[:,'gender2']
cansimTab2[['laborvar3','laborvar2','gender2']].head(50)

,laborvar3,laborvar2,gender2
0,populationTotal - Gender,population,Total - Gender
18,populationMen+,population,Men+
36,populationWomen+,population,Women+
54,labForceTotal - Gender,labForce,Total - Gender
99,labForceMen+,labForce,Men+
144,labForceWomen+,labForce,Women+
189,emplTotal - Gender,empl,Total - Gender
235,emplMen+,empl,Men+
280,emplWomen+,empl,Women+
325,emplFullTotal - Gender,emplFull,Total - Gender


### Reshape the data to the wide form
So far, data is still in a long form, vertically stacked.

Meanwhile, we only want three variables: time, value and variable mnemonics.

In [42]:
cansimpivot = cansimTab2.pivot(index='ref_date', columns='laborvar3', values='value')
cansimpivot.head(100)

laborvar3,emplFullMen+,emplFullTotal - Gender,emplFullWomen+,emplMen+,emplPartMen+,emplPartTotal - Gender,emplPartWomen+,emplRateMen+,emplRateTotal - Gender,emplRateWomen+,...,participRateWomen+,populationMen+,populationTotal - Gender,populationWomen+,unemplMen+,unemplRateMen+,unemplRateTotal - Gender,unemplRateWomen+,unemplTotal - Gender,unemplWomen+
ref_date,,,,,,,,,,,,,,,,,,,,,
1976-01,5712.1,8431.0,2718.9,6085.3,373.2,1205.6,832.4,73.0,57.2,41.7,...,45.4,8334.7,16852.4,8517.7,419.0,6.4,7.1,8.1,733.0,314.0
1976-02,5732.0,8467.5,2735.4,6097.0,364.9,1192.3,827.4,73.0,57.2,41.7,...,45.4,8353.3,16892.0,8538.8,416.2,6.4,7.0,8.1,730.0,313.8
1976-03,5758.5,8509.4,2751.0,6119.3,360.8,1194.8,833.9,73.1,57.3,41.9,...,45.3,8371.6,16930.7,8559.1,396.0,6.1,6.7,7.6,691.5,295.5
1976-04,5787.6,8540.1,2752.4,6145.3,357.6,1198.1,840.4,73.2,57.4,41.9,...,45.5,8390.0,16969.4,8579.4,402.0,6.1,6.8,8.0,713.1,311.1
1976-05,5770.4,8529.6,2759.3,6125.8,355.4,1196.4,841.0,72.9,57.2,41.9,...,45.6,8408.4,17008.1,8599.7,402.2,6.2,6.9,8.1,720.0,317.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1983-12,5912.3,9289.7,3377.4,6494.7,582.4,1897.8,1315.4,67.8,57.5,47.5,...,53.4,9574.7,19462.2,9887.5,843.8,11.5,11.3,11.1,1429.1,585.3
1984-01,5908.4,9297.8,3389.4,6487.7,579.3,1870.8,1291.5,67.7,57.3,47.3,...,53.2,9585.0,19483.7,9898.7,836.3,11.4,11.3,11.1,1423.3,586.9
1984-02,5927.1,9325.8,3398.7,6504.7,577.7,1878.1,1300.4,67.8,57.4,47.4,...,53.4,9595.8,19506.2,9910.4,829.2,11.3,11.3,11.3,1425.4,596.2


In [43]:
# replace strange values to NaN
cansimpivot.replace('..', np.nan)
cansimpivot.replace('x', np.nan)
cansimpivot.replace('F', np.nan)
cansimpivot.replace('M', np.nan)

laborvar3,emplFullMen+,emplFullTotal - Gender,emplFullWomen+,emplMen+,emplPartMen+,emplPartTotal - Gender,emplPartWomen+,emplRateMen+,emplRateTotal - Gender,emplRateWomen+,...,participRateWomen+,populationMen+,populationTotal - Gender,populationWomen+,unemplMen+,unemplRateMen+,unemplRateTotal - Gender,unemplRateWomen+,unemplTotal - Gender,unemplWomen+
ref_date,,,,,,,,,,,,,,,,,,,,,
1976-01,5712.1,8431.0,2718.9,6085.3,373.2,1205.6,832.4,73.0,57.2,41.7,...,45.4,8334.7,16852.4,8517.7,419.0,6.4,7.1,8.1,733.0,314.0
1976-02,5732.0,8467.5,2735.4,6097.0,364.9,1192.3,827.4,73.0,57.2,41.7,...,45.4,8353.3,16892.0,8538.8,416.2,6.4,7.0,8.1,730.0,313.8
1976-03,5758.5,8509.4,2751.0,6119.3,360.8,1194.8,833.9,73.1,57.3,41.9,...,45.3,8371.6,16930.7,8559.1,396.0,6.1,6.7,7.6,691.5,295.5
1976-04,5787.6,8540.1,2752.4,6145.3,357.6,1198.1,840.4,73.2,57.4,41.9,...,45.5,8390.0,16969.4,8579.4,402.0,6.1,6.8,8.0,713.1,311.1
1976-05,5770.4,8529.6,2759.3,6125.8,355.4,1196.4,841.0,72.9,57.2,41.9,...,45.6,8408.4,17008.1,8599.7,402.2,6.2,6.9,8.1,720.0,317.8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-08,9599.7,17149.9,7550.3,11052.8,1453.1,3804.9,2351.8,64.2,60.5,56.9,...,60.8,17228.9,34644.2,17415.3,909.7,7.6,7.1,6.5,1594.6,684.9
2025-09,9692.3,17256.0,7563.7,11103.6,1411.3,3759.3,2348.0,64.4,60.6,56.9,...,61.0,17241.8,34671.9,17430.1,881.4,7.4,7.1,6.8,1606.5,725.1
2025-10,9646.9,17237.5,7590.6,11125.2,1478.3,3844.4,2366.1,64.5,60.8,57.1,...,61.1,17252.7,34695.4,17442.7,854.7,7.1,6.9,6.6,1557.3,702.6


We can also reshape the data with two indices: ref_data and sex:

In [44]:
cansim_bySex = cansimTab2.pivot(index=('ref_date','gender'), columns='laborvar2', values='value')
cansim_bySex.head(100)

laborvar2                   empl  emplFull  emplPart  emplRate  labForce  \
ref_date gender                                                            
1976-01  Men+             6085.3    5712.1     373.2      73.0    6504.3   
         Total - Gender   9636.7    8431.0    1205.6      57.2   10369.7   
         Women+           3551.3    2718.9     832.4      41.7    3865.4   
1976-02  Men+             6097.0    5732.0     364.9      73.0    6513.1   
         Total - Gender   9659.8    8467.5    1192.3      57.2   10389.8   
...                          ...       ...       ...       ...       ...   
1978-08  Women+           3938.0    2971.0     967.0      43.6    4356.1   
1978-09  Men+             6344.7    5946.1     398.6      72.0    6870.3   
         Total - Gender  10305.5    8941.4    1364.1      57.7   11252.7   
         Women+           3960.8    2995.3     965.5      43.8    4382.4   
1978-10  Men+             6380.6    5978.4     402.2      72.3    6890.3   

laborvar2                participRate  population  unempl  unemplRate  
ref_date gender                                                        
1976-01  Men+                    78.0      8334.7   419.0         6.4  
         Total - Gender          61.5     16852.4   733.0         7.1  
         Women+                  45.4      8517.7   314.0         8.1  
1976-02  Men+                    78.0      8353.3   416.2         6.4  
         Total - Gender          61.5     16892.0   730.0         7.0  
...                               ...         ...     ...         ...  
1978-08  Women+                  48.3      9027.5   418.1         9.6  
1978-09  Men+                    78.0      8809.3   525.6         7.7  
         Total - Gender          63.0     17850.7   947.2         8.4  
         Women+                  48.5      9041.4   421.6         9.6  
1978-10  Men+                    78.1      8821.8   509.7         7.4  

[100 rows x 9 columns]

### Save the processed data file

In [45]:
# save to hard drive
fname = 'tbl' + cansimid +'_wide.csv'
cansimpivot.to_csv(fname)
fname = 'tbl' + cansimid +'_bySex.csv'
cansim_bySex.to_csv(fname)